# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane: Ranking Signal Analysis

I choose **Ranking Signal Analysis** because the lane guide asks which safe content and search signals are associated with visibility, clicks, engagement, or movement. I will use the starter dataset to compare performance across interpretable groups, measure associations and effect sizes, and produce a signal report that helps editors decide where to investigate first. The work will remain observational: it will identify useful patterns for decision-support, not prove that a signal causes a ranking change.

In [1]:
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen

import pandas as pd

DATA_CANDIDATES = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
local_path = next((path for path in DATA_CANDIDATES if path.exists()), None)
if local_path is not None:
    df = pd.read_csv(local_path)
    data_source = str(local_path)
else:
    data_url = "https://raw.githubusercontent.com/somnathsutra/ML-01-ASSIGNMENT/main/data/raw/content_refresh_anonymized.csv"
    with urlopen(data_url) as response:
        df = pd.read_csv(BytesIO(response.read()))
    data_source = data_url

print(f"Loaded {len(df):,} rows and {df.shape[1]:,} columns from the public-safe starter file")
print(f"Clients: {df['client_id'].nunique():,} | Content items: {df['content_id'].nunique():,}")
print(f"Content types: {df['content_type'].nunique():,}")

Loaded 30,000 rows and 44 columns from the public-safe starter file
Clients: 32 | Content items: 30,000
Content types: 3


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [2]:
decision_frame = {
    "decision": "Which content signal should an editor investigate first when reviewing visibility or engagement differences?",
    "actor": "A content editor or SEO analyst compares the signal report and chooses pages or groups for human review.",
    "action": "Prioritize diagnostic review, such as checking content depth, search demand, position, or engagement context.",
    "wrong_call_cost": "An editor may spend limited review time on a weak lead, or overlook a useful pattern; the analysis does not automatically trigger a content change.",
    "output": "A ranked signal report with grouped summaries, association sizes, and caveats.",
}
for name, statement in decision_frame.items():
    print(f"{name}: {statement}")

assert all(decision_frame.values())

decision: Which content signal should an editor investigate first when reviewing visibility or engagement differences?
actor: A content editor or SEO analyst compares the signal report and chooses pages or groups for human review.
action: Prioritize diagnostic review, such as checking content depth, search demand, position, or engagement context.
wrong_call_cost: An editor may spend limited review time on a weak lead, or overlook a useful pattern; the analysis does not automatically trigger a content change.
output: A ranked signal report with grouped summaries, association sizes, and caveats.


## 3. Quick look at the data (2-3 real numbers)

The starter snapshot is large enough for grouped signal comparisons: it contains 30,000 content items across 32 pseudonymized clients. It includes trailing-90-day performance totals, last-30-day versus previous-30-day movement fields, content metadata, CTR, average position, engagement, and scroll measures. These observed fields make a signal-analysis lane testable within the next seven weeks.

In [3]:
# Three descriptive numbers that motivate the lane.
summary = {
    "rows": len(df),
    "clients": df["client_id"].nunique(),
    "content_types": df["content_type"].nunique(),
    "declining_share": df["trend_direction"].astype("string").str.lower().eq("down").mean(),
    "median_impressions_90d": df["impressions_90d"].median(),
    "median_avg_position_with_data": df.loc[df["avg_position"] > 0, "avg_position"].median(),
}
print(f"Rows: {summary['rows']:,}")
print(f"Clients: {summary['clients']:,}")
print(f"Content types: {summary['content_types']:,}")
print(f"Observed decline-direction share: {summary['declining_share']:.3f}")
print(f"Median impressions over 90 days: {summary['median_impressions_90d']:,.1f}")
print(f"Median average position among rows with position data: {summary['median_avg_position_with_data']:.1f}")

assert summary["rows"] == 30_000
assert summary["clients"] == 32
assert summary["content_types"] == 3

Rows: 30,000
Clients: 32
Content types: 3
Observed decline-direction share: 0.542
Median impressions over 90 days: 731.0
Median average position among rows with position data: 11.4


## 4. Careful words: what I can and can't claim

I can claim that the data shows **observed** differences and **measured** associations between safe content/search signals and visibility, clicks, engagement, or movement. The results can support **directional** recommendations and a **decision-support** report for human review.

I cannot claim that a signal causes rankings, that it will work for every client, that a page will improve after editing, or that the analysis predicts Google's algorithm. The dataset is a pseudonymized snapshot with aggregated windows, patterned missingness, and no randomized intervention. I will keep identifiers out of features, treat `trend_direction` and `trend_pct` as label-related exclusions, and report effect sizes with uncertainty and limitations.

In [4]:
required_columns = {
    "content_id", "client_id", "content_type", "impressions_90d",
    "clicks_90d", "sessions_90d", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "trend_direction", "trend_pct",
}
missing_required = sorted(required_columns - set(df.columns))
print(f"Required signal fields present: {len(required_columns) - len(missing_required)}/{len(required_columns)}")
print(f"Missing required fields: {missing_required}")
print(f"Rows with measurable impressions: {(df['impressions_90d'] > 0).mean():.3f}")
print(f"Rows with position data: {(df['avg_position'] > 0).mean():.3f}")

assert not missing_required
assert (df["impressions_90d"] >= 0).all()
assert (df["sessions_90d"] >= 0).all()
assert df["trend_direction"].notna().all()
print("Research-question checks passed: the proposed signal report has the required observed fields.")

Required signal fields present: 12/12
Missing required fields: []
Rows with measurable impressions: 1.000
Rows with position data: 0.960
Research-question checks passed: the proposed signal report has the required observed fields.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.